# Lilly — train the Bosnian to English model

Set these in the panel on the right before running anything:

- **Session options → Accelerator → GPU** (P100 or T4; the notebook uses one either way)
- **Session options → Internet → On**
- **Input → Add Input → Datasets** → your uploaded copy of `models/lilly/translate`.
  Upload it once: Kaggle → Datasets → New Dataset → drag the folder in. The weights are
  too big for git, so this is how they reach the machine.

Then **Save Version → Save & Run All (Commit)** and close the tab. It keeps running
without you, about 2-3 hours, and the result waits in that version's **Output** tab.

Every step below stops the run if it fails, so a green version means it really worked.


In [ ]:
# 0. WHICH ARM THIS RUN TRAINS
# Set here, in the file that is committed, so the notebook that produced a build
# is the notebook in git. Editing this by hand before pushing is exactly the
# version-skew that killed a run at In [6]: the notebook went up from the
# working tree while the git clone below pulled a different script from GitHub.
#
#   "lora"    r=16 adapter, 4.3 M trainable parameters. The shipped recipe.
#   "fullft"  all 237.7 M parameters. training/PREREGISTRATION.md,
#             "v2 - read - read-egitim", written before this was ever run.
#
# Everything else is held identical between the two arms: same train-mix.tsv,
# same ntrex-holdout validation, same 2 epochs, same seed. Only the training
# mode changes, and the learning rate that the script resolves from it.
ARM = "fullft"
assert ARM in ("lora", "fullft")
print("arm:", ARM)


In [ ]:
# 1. Stop here unless the machine is actually set up
# Kaggle marks a version "complete" whenever no cell RAISES — a shell command that
# fails is not enough. So every check below is Python, and every later step is a
# checked subprocess. A misconfigured run should die in ten seconds, not in ten hours.
import os, subprocess, sys, urllib.error, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), (
    "No GPU. Right panel -> Session options -> Accelerator -> GPU, then Save & Run All again.")

# Pin to one GPU on purpose. With two visible, the trainer splits each batch across
# both, which doubles the effective batch and halves the optimizer steps while the
# learning rate stays put — a different recipe than the one these numbers were set for.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.cuda.device_count(), "GPU(s) visible, using:", torch.cuda.get_device_name(0))

def reachable(url):
    try:
        urllib.request.urlopen(url, timeout=20).close()
    except urllib.error.HTTPError:
        pass          # a status code still proves we got out
    except Exception as exc:
        raise SystemExit(
            f"Cannot reach {url} ({exc}). Right panel -> Session options -> Internet -> On.")

for host in ("https://github.com", "https://pypi.org", "https://object.pouta.csc.fi"):
    reachable(host)
print("network ok")

def run(*cmd):
    """Run a step and let a failure actually stop the notebook."""
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], check=True)


In [ ]:
# 2. Get the Lilly code
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", "Lilly"], check=True)
run("git", "clone", "-q", "https://github.com/ssaaffaakk/Lilly.git")
assert Path("/kaggle/working/Lilly/training").is_dir(), "clone produced nothing"
os.chdir("/kaggle/working/Lilly")
print("working in", os.getcwd())


In [ ]:
# 3. Install what we need (~2 min)
# The versions come out of the repo's own requirements.txt rather than being
# copied here. A second hand-kept list is how the speech run died: peft was
# pinned in requirements.txt and simply missing from that notebook's copy, so
# Kaggle's own much newer peft was used and its torchao dispatcher raised on the
# first get_peft_model() call — after a 3 GB download.
NEEDED = ["transformers", "peft", "accelerate", "sacrebleu", "sentencepiece",
          "sacremoses"]
pins = {}
for line in Path("requirements.txt").read_text(encoding="utf-8").splitlines():
    line = line.split("#")[0].strip()
    if "==" in line:
        pins[line.split("==")[0].strip().lower()] = line
unpinned = [n for n in NEEDED if n not in pins]
print("pinned here:", [pins[n] for n in NEEDED if n in pins])
print("no pin in requirements.txt, taking latest:", unpinned or "none")
run(sys.executable, "-m", "pip", "install", "-q", *[pins.get(n, n) for n in NEEDED])


In [ ]:
# 3b. Prove this machine can actually train, before hours are spent finding out
# Two things must hold that no version number shows: peft must be able to build a
# LoRA layer on this image, and the card Kaggle handed us must actually run
# kernels. A previous run satisfied "a GPU is available" on a P100 whose sm_60
# the installed PyTorch does not support, and failed only once it tried to
# compute — after the download.
import torch, torch.nn as nn
from peft import LoraConfig, get_peft_model
import peft, transformers
print("peft", peft.__version__, "| transformers", transformers.__version__)

class Tiny(nn.Module):
    def __init__(self):
        super().__init__()
        self.q_proj = nn.Linear(32, 32)
    def forward(self, x):
        return self.q_proj(x)

tiny = get_peft_model(Tiny(), LoraConfig(r=4, target_modules=["q_proj"]))
try:
    tiny = tiny.cuda()
    tiny(torch.randn(4, 32, device="cuda")).sum().backward()
except RuntimeError as exc:
    raise SystemExit(
        f"The GPU cannot run this build of PyTorch ({exc}).\n"
        f"Card: {torch.cuda.get_device_name(0)}. The accelerator is requested by "
        f"name in scripts/kaggle_train.py — it must be one of Kaggle's own "
        f"(NvidiaTeslaT4, NvidiaTeslaP100, ...), because an unrecognised name is "
        f"discarded silently and the run lands on the default card.") from exc

lora = [n for n, prm in tiny.named_parameters() if "lora_" in n and prm.grad is not None]
assert lora, "peft built a LoRA layer but no gradient reached it"
print(f"LoRA trains on {torch.cuda.get_device_name(0)}: "
      f"{len(lora)} adapter tensors took a gradient")
del tiny
torch.cuda.empty_cache()


In [ ]:
# 4. Find the base weights and the pinned corpus among the datasets you attached
import glob
found = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/source.spm", recursive=True)]
assert found, ("Attach your models/lilly/translate folder as a Dataset input "
               "(right panel -> Input -> Add Input -> Datasets)")
assert os.path.exists(os.path.join(found[0], "config.json")), f"{found[0]} looks incomplete"
os.environ["LILLY_BASE"] = found[0]
print("base weights:", found[0])

# The extra corpus is attached, not re-harvested. build_training_mix.py asserts
# every per-step count exactly, and those counts were measured on one specific
# 38,277-row file. A fresh download_extra_data.py run on this machine returned
# 38,280 instead -- a live web corpus does not come back byte-identical -- and
# three rows is enough to kill the run on its first assertion. Pinning the file
# is what makes the recipe's arithmetic mean anything here.
corpus = sorted(glob.glob("/kaggle/input/**/extra-train.tsv", recursive=True))
assert corpus, ("Attach the Lilly extra corpus as a Dataset input. "
                "scripts/kaggle_train.py uploads it as <user>/lilly-extra-corpus "
                "and attaches it automatically -- launch from the laptop with "
                "`python3 scripts/kaggle_train.py translation` rather than "
                "pushing this notebook by hand.")
CORPUS = corpus[0]
print("pinned corpus:", CORPUS)


In [ ]:
# 5. Download and clean the Bosnian-English data (~12 min)
import shutil

def rows_in(path):
    with open(path, encoding="utf-8") as fh:
        return sum(1 for _ in fh)

run("python3", "data/scripts/download_data.py")
run("python3", "data/scripts/clean_data.py")

# WikiMatrix is web-mined and about a ninth of it is not a translation pair at
# all — the two sides are merely about the same subject. Three rules, each
# measured against SETIMES and TED2020 (aligned by construction, so whatever
# fires there is the false-positive floor: 0.14%), drop 21,178 of them. 132
# pairs were read by hand across three rounds to calibrate. Left in, those pairs
# teach the model to invent.
run("python3", "data/scripts/filter_train_data.py")

# data/clean/train.tsv means exactly one thing from here on: the filter's output.
# It is never appended to. The previous version of this cell concatenated the
# extra corpus onto it, so the file's contents depended on how far the notebook
# had run -- and build_training_mix.py, which reads train.tsv AND the extra file,
# counted every extra pair twice: 390,172 rows against an expected 351,889. The
# mix builder is the single place that combines the two.
filtered = rows_in("data/clean/train.tsv")
assert filtered == 313_612, (
    f"the filter produced {filtered:,} rows, expected 313,612. Every count in "
    f"build_training_mix.py was measured on that number.")

# 38,277 pairs the base model has never seen: wikimedia-v20260327 and the
# professionally translated NTREX-128, copied from the attached dataset.
Path("data/extra").mkdir(parents=True, exist_ok=True)
shutil.copyfile(CORPUS, "data/extra/extra-train.tsv")
extra = rows_in("data/extra/extra-train.tsv")
assert extra == 38_277, f"the pinned corpus has {extra:,} rows, expected 38,277"

# The benchmark the base model has never seen. Our own split came from the same
# corpora it was trained on, so a gain there would prove nothing on its own.
run("python3", "data/scripts/download_flores.py")

# ---- leakage, measured here rather than inherited --------------------------
# download_extra_data.py deduplicates against valid/test by WHOLE PAIR and
# against FLORES by EITHER SIDE. Not re-harvesting skips that, so both inputs
# are re-checked here, before anything trains on them.
#
# The counts below are what this corpus actually contains, measured, not assumed:
#
#   data/clean/train.tsv        313,612 rows   whole-pair 0   FLORES 0   one-side 8
#   data/extra/extra-train.tsv   38,277 rows   whole-pair 0   FLORES 0   one-side 1
#
# Whole-pair overlap is the one that would let the model memorise a held-out
# answer, and it is zero. FLORES is the set the pre-registered decision is
# measured on, and it is zero. The nine one-side rows are citation boilerplate
# sharing one side with one held-out row -- "Aftenposten (in Norwegian)." against
# test.tsv line 1348 is the extra corpus's only one -- and eight of the nine are
# in the corpus the shipped model already trained on. They are pinned at their
# measured counts instead of asserted to zero: asserting zero would be asserting
# something that was never true of this corpus, and pinning catches a real jump
# while leaving a known, quantified 0.0026% visible rather than hidden.
def norm(s):
    return " ".join(s.lower().split())

held_side, held_pair = set(), set()
for split in ("data/clean/valid.tsv", "data/clean/test.tsv"):
    with open(split, encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) == 3:
                held_side.add(norm(parts[1]))
                held_side.add(norm(parts[2]))
                held_pair.add((norm(parts[1]), norm(parts[2])))
flores = set()
for path in sorted(glob.glob("data/flores/*")):
    if os.path.isfile(path):
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                flores.add(norm(line))

for path, expect_side in (("data/clean/train.tsv", 8),
                          ("data/extra/extra-train.tsv", 1)):
    side = pair = flo = 0
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            parts = line.rstrip("\n").split("\t")
            if len(parts) != 3:
                continue
            nb, ne = norm(parts[1]), norm(parts[2])
            if (nb, ne) in held_pair:
                pair += 1
            if nb in flores or ne in flores:
                flo += 1
            if nb in held_side or ne in held_side:
                side += 1
    print(f"  {path}: whole-pair {pair}, FLORES {flo}, one-side {side}")
    assert pair == 0, f"{path} shares {pair} whole pairs with valid/test"
    assert flo == 0, f"{path} shares {flo} lines with FLORES — the decision set"
    assert side == expect_side, (
        f"{path} one-side overlap moved: {side}, was {expect_side} when measured")

print(f"{filtered:,} filtered + {extra:,} pinned = {filtered + extra:,} pairs into the mix")


In [ ]:
# 5b. Rebuild the corpus so its length variance matches the benchmark's
# The fine-tuning's chrF2 cost is entirely its terseness, and the cause is not
# the corpus mean — that already matches FLORES to 0.3% — but its variance:
# WikiMatrix carries sd(log char ratio) 0.213 against FLORES's 0.129, so the
# model learns from a much wider spread of length ratios than it is asked to
# produce. This splits to one sentence per example inside a length band tilted
# slightly long, and holds 462 ntrex pairs out for validation.
run("python3", "data/scripts/build_training_mix.py")
mix_rows = sum(1 for _ in open("data/clean/train-mix.tsv", encoding="utf-8"))
holdout = sum(1 for _ in open("data/clean/ntrex-holdout.tsv", encoding="utf-8"))
print(f"{mix_rows:,} training examples, {holdout:,} held out")
assert mix_rows > 340_000, f"only {mix_rows:,} examples — the mix build fell short"


In [ ]:
# 6. Quick pipeline check (~3 min) — a toy run, just to prove everything works.
# It writes to models/quicktest-adapter, never to the real one.
run("python3", "training/train_translation.py", "--quick-test")


In [ ]:
# 7. THE REAL TRAINING (~3-4 hours)
# Two epochs, not one. The shipped adapter was a single epoch because this
# script's --epochs default is 1.0 and the notebook called it with no arguments;
# that was never a decision, it was a default nobody looked at. On the rebuilt
# mix the validation set is the 462 held-out ntrex pairs — professionally
# translated, and never in the training corpus — rather than 500 rows of a split
# drawn from the same corpora the base model already saw.
common = ["--data", "data/clean/train-mix.tsv",
          "--valid", "data/clean/ntrex-holdout.tsv",
          "--valid-limit", "0",
          "--epochs", "2"]

if ARM == "fullft":
    # One real optimiser step first, printing peak VRAM, then stop. Full
    # fine-tuning goes from 4.3 M trainable parameters to 237.7 M, and the Adam
    # states alone are ~1.9 GB on top of gradients and activations. Learning
    # that it does not fit costs a minute here and three hours in the middle of
    # the run. --preflight returns before it builds the real dataset.
    run("python3", "training/train_translation.py", "--full-finetune",
        "--preflight", *common)
    run("python3", "training/train_translation.py", "--full-finetune", *common)
    assert Path("models/lilly/translate-fullft/config.json").is_file(), \
        "no full fine-tuned weights were written"
else:
    run("python3", "training/train_translation.py", *common)
    assert Path("models/lilly/adapter/adapter_config.json").is_file(), \
        "no adapter was written"


In [ ]:
# 8. Score it: base vs Lilly, on our split AND on FLORES-200 (~20 min)
# FLORES-200 is the number that counts. read-veri checked that properly rather
# than assuming it: FLORES appears in no corpus of the base model's own
# opusTCv20210807 manifest, and 0 of its 2,009 Bosnian sentences were found by a
# sentence-level sweep of that training set.
#
# evaluate.py loads a base and applies an adapter to it, so it can only score the
# LoRA arm. The full fine-tune's deciding number is produced at home by
# training/evaluate_app.py, through app.translate.Engine, which is the product
# path and the only one PREREGISTRATION.md accepts. Scoring it here as well
# would be a second, different measurement of one model — and three different
# answers for one fine-tune is a mistake this project has already made.
if ARM == "fullft":
    print("full fine-tune arm: scored at home through app.translate.Engine.")
    print("  python3 scripts/build_translator.py --source models/lilly/translate-fullft \\")
    print("      --dest models/lilly/translator-fullft --no-adapter")
    print("  python3 training/evaluate_app.py --tuned models/lilly/translator-fullft")
else:
    run("python3", "training/evaluate.py", "--adapter", "models/lilly/adapter")
    print(Path("training/RESULTS.md").read_text())


In [ ]:
# 9. Package the result so it survives the run
if ARM == "fullft":
    # Whole weights, not an adapter: ~950 MB of fp32 before compression.
    run("zip", "-qr", "/kaggle/working/lilly-fullft.zip",
        "models/lilly/translate-fullft")
    out, floor = Path("/kaggle/working/lilly-fullft.zip"), 200_000_000
else:
    assert Path("training/RESULTS.md").is_file()
    run("zip", "-qr", "/kaggle/working/lilly-adapter.zip",
        "models/lilly/adapter", "training/RESULTS.md")
    out, floor = Path("/kaggle/working/lilly-adapter.zip"), 1_000_000

size = out.stat().st_size
assert size > floor, f"{out.name} is only {size} bytes — the save did not happen"
print(f"{out.name} — {size / 1048576:.1f} MB, in the Output tab when this finishes")


**Done.** Download `lilly-adapter.zip` from this version's **Output** tab and unzip it
so the adapter sits at `models/lilly/adapter/`. The app uses it the next time it starts.

Read `RESULTS.md` first. If the tuned row is not above the base row, the run did not help
and there is nothing worth shipping — more epochs or more data before another attempt.
